# Get To Know A Dataset: END Sea Surface Temperature Collections

This tutorial introduces END's versioned conversions of six upstream SST analyses.
It follows the topics in the [AWS tutorial template](https://github.com/awslabs/open-data-docs/tree/main/get-to-know-a-dataset).
The [dataset guide](README.md) describes coverage, fields and quality limits.

**Public hosting is not configured yet.** Fill the configuration below from the
validated public release manifest after account linkage and publication. This
notebook uses anonymous reads by default. It will run once the first public
release is available and its endpoint and snapshot have been configured.
The Registry landing-page link will be added after AWS creates the entry.

Install the accompanying `requirements.txt` in Python 3.12, then open this notebook.


## Find the data

The proposed public bucket separates `data/<collection>.icechunk/` repositories,
`metadata/` release and quality records, and `docs/`. A release manifest gives the
bucket, region, prefix and fixed snapshot for each collection. Use those values
instead of a mutable branch when reproducing results. `coverage.json` is a
pre-release coverage inventory, not that public release manifest.

CMC V2 is historical; CMC V3, GAMSSA, GPB05 night, OISST and OSTIA are active feeds.
This example uses OISST for October 11–17, 2026, in the equatorial Pacific.


In [ ]:
# Replace only with values from the validated public release manifest.
BUCKET = None
REGION = "us-west-2"
PREFIX = "data/oisst.icechunk"
SNAPSHOT_ID = None
ANONYMOUS = True
START_DATE = "2026-10-11"
END_DATE = "2026-10-17"

if not BUCKET or not SNAPSHOT_ID:
    raise ValueError("Configure BUCKET and SNAPSHOT_ID from the public release manifest.")


## Understand the format

[Zarr V3](https://zarr.readthedocs.io/en/stable/) stores chunked arrays.
[Icechunk](https://icechunk.io/en/latest/guides/storage/) adds transactional commits
and immutable snapshots. Xarray labels dimensions and decodes CF units, packing
and missing values; Dask loads only chunks needed for the selected computation.
Open the repository through Icechunk rather than treating its directory as a
standalone Zarr mapping.

The example uses S3 for reads. Compute runs on your laptop or in your own AWS
account; colocating compute with the public bucket can reduce transfer latency.
Do not run analysis compute in the sponsored hosting account.


In [ ]:
import importlib.metadata
import time

import dask
import icechunk
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

versions = {name: importlib.metadata.version(name)
            for name in ("icechunk", "zarr", "xarray", "dask", "numpy")}
print(versions)
started = time.perf_counter()
storage = icechunk.s3_storage(
    bucket=BUCKET, prefix=PREFIX, region=REGION, anonymous=ANONYMOUS,
)
repository = icechunk.Repository.open(storage)
session = repository.readonly_session(snapshot_id=SNAPSHOT_ID)
with xr.open_zarr(session.store, consolidated=False, chunks={},
                  mask_and_scale=True, decode_times=True) as dataset:
    print(dict(dataset.sizes))
    print({name: dataset[name].attrs.get("units") for name in ("sst", "anom")})
    # Apply the documented last-row-per-calendar-date view to legacy revisions.
    days = dataset.time.values.astype("datetime64[D]")
    _, reverse_indices = np.unique(days[::-1], return_index=True)
    keep = np.sort(len(days) - 1 - reverse_indices)
    daily = dataset.isel(time=keep).sortby("time")
    window = daily[["sst", "anom", "preliminary"]].sel(
        time=slice(START_DATE, END_DATE), lat=slice(-5, 5), lon=slice(190, 240),
    ).isel(zlev=0, drop=True)
    expected_days = np.arange(np.datetime64(START_DATE),
                              np.datetime64(END_DATE) + np.timedelta64(1, "D"))
    if not np.array_equal(window.time.values.astype("datetime64[D]"), expected_days):
        raise ValueError("The selected release does not contain every requested day.")
    if not window.sizes["lat"] or not window.sizes["lon"]:
        raise ValueError("The requested region is empty; inspect coordinate conventions.")
    if window.sst.attrs.get("units", "").lower() not in ("celsius", "degrees_celsius", "degc"):
        raise ValueError("Expected decoded OISST temperatures in Celsius.")
    with dask.config.set(scheduler="threads", num_workers=2):
        subset = window.compute()

print({"snapshot": SNAPSHOT_ID, "decoded_subset_bytes": subset.nbytes,
       "seconds": round(time.perf_counter() - started, 2)})
# Decoded subset bytes are memory size, not compressed S3 bytes transferred.
print("Preliminary by date:", list(zip(subset.time.values.astype("datetime64[D]").astype(str),
                                      subset.preliminary.values.astype(bool))))


## Inspect and plot a bounded subset

OISST's `sst` and `anom` are already decoded to Celsius. Its singleton depth
coordinate was removed for this surface example. Missing packed values decode
to NaN; they remain missing in the map and regional mean. The source's anomaly
uses its own climatology (1971–2000 in the inspected input), so this is not an
anomaly rebased to a different baseline.


In [ ]:
latest = subset.sst.isel(time=-1)
print("Valid SST cells by day:", subset.sst.notnull().sum(("lat", "lon")).values.tolist())
fig, ax = plt.subplots(figsize=(10, 3.5))
latest.plot(ax=ax, cmap="viridis", cbar_kwargs={"label": "SST (°C)"})
ax.set_title(f"OISST SST on {END_DATE}; preliminary={bool(subset.preliminary.values[-1])}")
ax.set_xlabel("Longitude (degrees east)")
ax.set_ylabel("Latitude (degrees north)")
plt.show()


## Answer a small research question

How does mean SST anomaly in 5°S–5°N, 170°W–120°W vary over this week?
Weight cells by cosine of latitude because a regular latitude–longitude grid
has unequal cell areas. The weights are renormalized over nonmissing cells.
This short example is not an official ENSO index or a climate trend estimate.
Longer analyses should declare baseline, date coverage, preliminary/final policy
and missing-value handling before interpreting changes.


In [ ]:
weights = np.cos(np.deg2rad(subset.lat))
regional = subset.anom.weighted(weights).mean(("lat", "lon"), skipna=True)
if not np.isfinite(regional.values).all():
    raise ValueError("No valid anomaly estimate for one or more selected dates.")
print(regional.to_series())
fig, ax = plt.subplots(figsize=(8, 3))
regional.plot(ax=ax, marker="o")
ax.axhline(0, color="grey", linewidth=0.7)
ax.set_title("Equatorial Pacific regional mean OISST anomaly")
ax.set_ylabel("Anomaly (°C; source climatology)")
plt.show()


## Community challenge

Extend the example across several years and compare two source analyses over
the same ocean region. Quantify how grid resolution, preliminary revisions,
missing dates and differing SST definitions affect the result. Start with daily
regional reductions rather than downloading or regridding the entire global
archive. Retain both source snapshots and report the dates and area actually
compared. For OSTIA, resolve coordinate partitions before combining dates.

A useful contribution would include a reproducible notebook, sensitivity to
missing-data rules, and measured runtime and S3 transfer volume. Do not treat
agreement between two analyses as independent validation of their shared inputs.

## Attribution and reproducibility

Cite NOAA/NCEI's [OISST product](https://www.ncei.noaa.gov/products/optimum-interpolation-sst)
and its recommended scientific references, plus the END snapshot, selected dates,
region, dependency versions and access date. See [source rights](source-rights.md)
for the other collections. For conversion or example problems, use
[GitHub issues](https://github.com/endcorp-hq/open-data-examples/issues).
The Registry link will be added after launch. Sponsorship is not claimed active.
